# BigAlpha E5H0：共享 E1 日内编码器 + 单层日级 GRU + 同日 1 分钟残差

`main` 只加载训练好的 `ba_model.json` 并推理。5 个交易日分别复用 E1 的 5 分钟日内编码器，固定算子逐日重置；5 个日向量进入单层日级 GRU。没有 day position、跨日注意力或连续 240 步算子。1 分钟残差分支保持 E4A 原结构且不含循环层。纯推理不需要 `init_e4a_0850.json`。

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch

from ba_common import (
    ResidualFrequencyPanelTensor, align_panel_to_master,
    apply_datasources, apply_universe_mask, build_model, build_panel,
    config_for_frequency, load_ckpt_json, load_universe, now_s,
    predict_scores, set_seed,
)
from ba_train import BASE_MODEL_PATH, MODEL_PATH, train_and_save


def main(datasources, start_date, end_date) -> pd.DataFrame:
    """用冻结的共享日内/day-GRU 5m 主干与原 1m 增量生成分数。"""
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到 {MODEL_PATH}；请先运行 train_e5h0.ipynb 生成模型。"
        )

    ckpt = load_ckpt_json(MODEL_PATH)
    config = apply_datasources(dict(ckpt["config"]), datasources)
    expected = {
        "model_version": "raw_hier_1m_res_v1",
        "experiment_id": "E5H0_SHARED_DAY_GRU_1M_RESIDUAL",
        "memory_pipeline_version": "staged_5m_release_then_1m_monthly_v1",
        "lookback_days": 5,
        "v2_long_window_mode": "shared_day_gru_residual",
        "v2_long_position_mode": "shared_intraday_only",
        "v2_day_gru_hidden": 48,
        "v2_day_gru_layers": 1,
        "v2_train_context_only": True,
        "v2_use_crossday": False,
    }
    bad = {k: (config.get(k), v) for k, v in expected.items()
           if config.get(k) != v}
    if bad:
        raise ValueError(f"ba_model.json 不是 E5H0 受控版本: {bad}")

    audit = ckpt.get("meta", {}).get("architecture_audit", {})
    required_audit = {
        "operator_reset_each_day": True,
        "shared_intraday_encoder": True,
        "day_gru_layers": 1,
        "day_gru_hidden": 48,
        "day_position_encoding": False,
        "crossday_attention": False,
        "one_minute_residual_rnn_absent": True,
    }
    audit_bad = {k: (audit.get(k), v) for k, v in required_audit.items()
                 if audit.get(k) != v}
    if audit_bad:
        raise ValueError(f"checkpoint 结构审计不完整: {audit_bad}")
    if audit.get("per_day_e1_embedding_max_abs_diff", 1.0) > 2e-5:
        raise ValueError("共享日内编码器未严格复现逐日 E1 表示")
    if audit.get("initial_e1_max_abs_diff", 1.0) > 2e-5:
        raise ValueError("训练时 epoch0 未严格复现 E1")

    state_keys = set(ckpt["state_dict"])
    for prefix in ("base.day_gru.", "base.day_delta_head."):
        if not any(k.startswith(prefix) for k in state_keys):
            raise ValueError(f"checkpoint 缺少 {prefix} 权重")
    forbidden = [k for k in state_keys
                 if k in {"base.day_pos", "base.fixed_day_sincos"}
                 or k.startswith(("base.day_encoder.", "base.day_norm.",
                                  "base.crossday_pool."))]
    if forbidden:
        raise ValueError(f"checkpoint 混入旧跨日模块: {forbidden[:8]}")
    residual_rnn_keys = [k for k in state_keys if k.startswith("residual.")
                         and ("gru" in k.lower() or "lstm" in k.lower())]
    if residual_rnn_keys:
        raise ValueError(f"1m残差分支混入循环层: {residual_rnn_keys[:8]}")

    fm = ckpt.get("frequencies", {})
    if not {"5m", "1m"}.issubset(fm):
        raise ValueError("checkpoint 缺少 5m/1m 频率元数据")

    set_seed(config["seed"], config.get("deterministic_training", False))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    start_date, end_date = str(start_date)[:10], str(end_date)[:10]
    lookback_days = int(config["lookback_days"])
    history_start = (pd.Timestamp(start_date) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
    print(f"[{now_s()}] E5H0推理 [{start_date}, {end_date}]，"
          f"5m历史从 {history_start} 读取 | device={device}")

    universe = load_universe(config, history_start, end_date)
    if universe:
        source_days = sorted(universe)
        source_codes = sorted(set().union(*universe.values()))
    else:
        source_days = source_codes = None

    panel5 = build_panel(
        config_for_frequency(config, "5m"), history_start, end_date,
        fields=fm["5m"]["fields"], bar_slots=fm["5m"]["bar_slots"],
        instruments=source_codes, master_days=source_days,
        master_codes=source_codes,
    )
    panel1 = build_panel(
        config_for_frequency(config, "1m"), history_start, end_date,
        fields=fm["1m"]["fields"], bar_slots=fm["1m"]["bar_slots"],
        instruments=source_codes, master_days=source_days,
        master_codes=source_codes,
    )
    panel1 = align_panel_to_master(panel1, panel5["days"], panel5["codes"])
    apply_universe_mask(panel5, universe, mask_history=False)
    apply_universe_mask(panel1, universe, mask_history=False)

    panels = {"5m": panel5, "1m": panel1}
    norms = {
        "5m": (fm["5m"]["norm_mu"], fm["5m"]["norm_sd"]),
        "1m": (fm["1m"]["norm_mu"], fm["1m"]["norm_sd"]),
    }
    panel_tensor = ResidualFrequencyPanelTensor(
        panels, norms, config, device, keep_on_device=False
    )

    model = build_model(
        config, ckpt["fields"], ckpt["bars_per_day"],
        ckpt["bar_slots"], device, frequency_meta=fm,
    )
    if model.base.day_gru.num_layers != 1 or model.base.day_gru.hidden_size != 48:
        raise RuntimeError("推理模型 day-GRU 结构异常")
    residual_rnns = [name for name, module in model.residual.named_modules()
                     if isinstance(module, (torch.nn.GRU, torch.nn.LSTM))]
    if residual_rnns:
        raise RuntimeError(f"推理模型1m分支含循环层: {residual_rnns}")
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.freeze_base().eval()

    days_arr = np.asarray(panel5["days"]).astype(str)
    eval_ids = np.where((days_arr >= start_date) & (days_arr <= end_date))[0]
    if len(eval_ids) == 0:
        raise RuntimeError("评估区间与5m panel没有交集")
    if int(eval_ids[0]) < lookback_days - 1:
        raise RuntimeError(
            f"评估首日前只有 {int(eval_ids[0])} 个交易日，"
            f"不足 lookback_days={lookback_days}，拒绝零填充推理"
        )

    code_pos = {c: i for i, c in enumerate(panel5["codes"].tolist())}
    rows, t0 = [], time.time()
    with torch.no_grad():
        for d in eval_ids.tolist():
            day = str(panel5["days"][d])
            if universe is not None and day in universe:
                codes = sorted(universe[day])
            else:
                codes = panel5["codes"][panel5["valid"][d]].tolist()
            codes = [c for c in codes if c in code_pos]
            if not codes:
                print(f"[warn] {day}: universe 与 panel 无交集")
                continue
            idx = np.asarray([code_pos[c] for c in codes], dtype=np.int64)
            batch = panel_tensor.make(d, idx, include_base=True)
            score = predict_scores(model, batch, config["predict_chunk"])
            rows.append(pd.DataFrame({
                "date": day,
                "instrument": codes,
                "score": score.float().cpu().numpy(),
            }))

    if not rows:
        raise RuntimeError("评估区间内没有生成任何分数")
    result = pd.concat(rows, ignore_index=True)
    result["date"] = pd.to_datetime(result["date"])
    result["score"] = result["score"].astype(np.float64)
    result = (
        result.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    n_days = result["date"].nunique()
    print(f"[{now_s()}] 完成 {len(result)} 行 / {n_days} 日，"
          f"日均 {len(result) / max(n_days, 1):.0f} 股 | "
          f"{time.time() - t0:.0f}s")
    return result

In [ ]:
if __name__ == "__main__":
    from bigmodule import M

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "bar5m": "bigalpha_2026_stock_bar5m",
    }
    if not os.path.exists(MODEL_PATH):
        print(f"未发现 {MODEL_PATH}，开始训练 E5H0")
        train_and_save(
            datasources=datasources,
            out_path=MODEL_PATH,
            base_model_path=BASE_MODEL_PATH,
        )

    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)